[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/072.%20The%20Capacitated%20VRP%20%28CVRP%29/P72-Tier-7_executed.ipynb)

---

# 72. The Capacitated VRP (CVRP)

## Tier 7: Human–AI Partnership

### Purpose

This notebook demonstrates a small simulation of human–AI collaboration around capacitated vehicle-routing decisions. It represents experts, an AI recommendation system, confidence-based intervention, trust updates, feedback, and partnership metrics. The simulation is intended to make the interaction logic inspectable; it is not an empirical study or a production routing system.

### Inputs and outputs

The inputs are synthetic customer locations and demands, vehicle capacities and locations where provided, and fixed model parameters. The notebook produces heuristic routing or assignment recommendations, records simulated human–AI decisions, summarizes the interaction metrics, and visualizes the resulting records.

### Scope and modeling assumptions

- Locations are two-dimensional coordinates, and distances are Euclidean distances in coordinate units.
- Route planning uses a nearest-feasible-customer heuristic for one vehicle. It does not guarantee an optimal route and may leave customers unserved when the vehicle reaches its capacity.
- Vehicle assignment processes customers in input order and assigns each to the nearest vehicle with sufficient remaining capacity. Its distance calculation is an open path from each vehicle's starting location through its assigned customers; it does not include a return leg.
- The general decision case reports the route-planning and vehicle-assignment subresults separately. Their distances are not added, because the subresults may describe overlapping customers.
- Human intervention, confidence, trust, and AI feedback are generated by simple rules and random draws. They are illustrative model quantities, not measured human behavior, calibrated probabilities, or verified solution quality.
- This notebook does not model time windows, travel-time uncertainty, service times, road networks, or a human-only or AI-only benchmark. A performance improvement over such a benchmark therefore cannot be inferred from these simulations.

### Simulation sequence

1. Represent experts, AI capabilities, decisions, and partnership metrics.
2. Generate an AI recommendation with a heuristic, a rule-based confidence score, and an explanation.
3. Let the selected expert accept or intervene according to trust, confidence, and explanation-quality rules.
4. Record the final decision and update the model's trust and feedback proxies.
5. Summarize and visualize the simulated interactions.

### Routing notation

For a single vehicle, let \(0\) denote the depot and let \(C\) be the set of customers. Customer \(i\) has location \(p_i=(x_i,y_i)\) and nonnegative demand \(q_i\). The vehicle has capacity \(Q\). The Euclidean distance between locations \(i\) and \(j\) is

$$
d_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}.
$$

A complete route is a sequence beginning and ending at the depot, serving each assigned customer once, and satisfying the capacity constraint

$$
\sum_{i\in C}q_i\leq Q.
$$

Its distance is the sum of the distances between consecutive locations in the sequence. In this notebook, the route heuristic selects the nearest unvisited customer that fits the remaining capacity. If no remaining customer fits, it stops and reports the customers it did not serve. Thus, the heuristic constructs a feasible partial route when the entire set cannot fit; it does not claim to solve the full CVRP optimally.

### Human–AI interaction notation

Let \(a_t\) be the AI recommendation at decision \(t\), \(c_t\in[0,1]\) its rule-based confidence score, and \(h_t\in\{0,1\}\) indicate whether the expert intervenes. Trust is stored separately for each expert and updated after each interaction. The simulation's agreement indicator is \(1-h_t\); it measures acceptance of the recommendation, not whether the resulting route is objectively correct. The code below implements these rules explicitly.

In [ ]:
from dataclasses import dataclass
from enum import Enum
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# Fixed seeds make the synthetic scenarios and interaction draws reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported for the human–AI partnership simulation.")

In [ ]:
class ExpertiseLevel(Enum):
    NOVICE = "novice"
    INTERMEDIATE = "intermediate"
    EXPERT = "expert"
    MASTER = "master"


class DecisionType(Enum):
    ROUTE_PLANNING = "route_planning"
    VEHICLE_ASSIGNMENT = "vehicle_assignment"
    PRIORITY_ADJUSTMENT = "priority_adjustment"
    CONSTRAINT_HANDLING = "constraint_handling"


@dataclass
class HumanExpert:
    """A simulated logistics expert with domain-specific decision patterns."""

    expert_id: int
    name: str
    expertise_level: ExpertiseLevel
    years_experience: int
    specializations: List[str]
    decision_patterns: Dict[str, float]
    confidence_threshold: float
    trust_score: float = 0.5

    def __post_init__(self) -> None:
        self.decision_history: List[Any] = []
        self.ai_interactions: List[Any] = []
        self.learning_rate = 0.1


@dataclass
class AIOptimizer:
    """Stores the simulated AI system's settings and feedback proxies."""

    ai_id: str
    optimization_method: str
    confidence_calibration: float
    explanation_capability: float
    learning_capability: float

    def __post_init__(self) -> None:
        self.decision_history: List[Any] = []
        self.human_feedback: List[Any] = []
        self.performance_metrics = {
            "accuracy": 0.8,
            "explainability": 0.7,
            "adaptability": 0.6,
        }


@dataclass
class CollaborativeDecision:
    """A record of one simulated human–AI decision."""

    decision_id: str
    timestamp: float
    problem_context: Dict[str, Any]
    ai_recommendation: Dict[str, Any]
    ai_confidence: float
    ai_explanation: str
    human_input: Optional[Dict[str, Any]]
    human_confidence: float
    final_decision: Dict[str, Any]
    agreement_level: float
    decision_type: DecisionType
    explanation_quality: float


@dataclass
class PartnershipMetrics:
    """Summary metrics for the recorded interaction history."""

    total_decisions: int
    agreement_rate: float
    trust_evolution: List[float]
    performance_improvement: Optional[float]
    learning_effectiveness: float
    explanation_quality: float
    decision_time: Optional[float]
    satisfaction_score: float


print("Human–AI partnership data structures defined.")

In [ ]:
class HumanExpertSystem:
    """Models experts and their rule-based evaluation of AI recommendations."""

    def __init__(self, num_experts: int = 3):
        self.num_experts = num_experts
        self.experts: List[HumanExpert] = []
        self._initialize_experts()

    def _initialize_experts(self) -> None:
        """Create the configured experts and their domain-confidence patterns."""
        expert_configs = [
            {
                "name": "Sarah Chen",
                "level": ExpertiseLevel.EXPERT,
                "experience": 12,
                "specializations": ["route_optimization", "capacity_planning"],
                "confidence_threshold": 0.7,
            },
            {
                "name": "Michael Rodriguez",
                "level": ExpertiseLevel.MASTER,
                "experience": 18,
                "specializations": ["vehicle_coordination", "emergency_response"],
                "confidence_threshold": 0.8,
            },
            {
                "name": "Emily Johnson",
                "level": ExpertiseLevel.EXPERT,
                "experience": 10,
                "specializations": ["customer_service", "time_windows"],
                "confidence_threshold": 0.75,
            },
        ]

        pattern_keys = {
            DecisionType.ROUTE_PLANNING.value: "route_optimization",
            DecisionType.VEHICLE_ASSIGNMENT.value: "vehicle_coordination",
            DecisionType.PRIORITY_ADJUSTMENT.value: "customer_service",
            DecisionType.CONSTRAINT_HANDLING.value: "capacity_planning",
        }

        for index, config in enumerate(expert_configs[:self.num_experts]):
            patterns = {}
            for decision_key, specialization in pattern_keys.items():
                strong_match = specialization in config["specializations"]
                patterns[decision_key] = 0.85 if strong_match else 0.65

            self.experts.append(
                HumanExpert(
                    expert_id=index + 1,
                    name=config["name"],
                    expertise_level=config["level"],
                    years_experience=config["experience"],
                    specializations=config["specializations"],
                    decision_patterns=patterns,
                    confidence_threshold=config["confidence_threshold"],
                )
            )

    def evaluate_ai_recommendation(
        self,
        expert: HumanExpert,
        ai_recommendation: Dict[str, Any],
        ai_confidence: float,
        ai_explanation: str,
        problem_context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Apply the simulation's confidence, trust, and intervention rules."""
        decision_type = problem_context.get(
            "decision_type", DecisionType.ROUTE_PLANNING
        )
        domain_confidence = expert.decision_patterns.get(decision_type.value, 0.5)
        explanation_quality = self._evaluate_explanation(ai_explanation, expert)

        confidence_alignment = 1.0 - abs(ai_confidence - domain_confidence)
        acceptance_probability = (
            expert.trust_score * 0.4
            + confidence_alignment * 0.3
            + explanation_quality * 0.3
        )

        if ai_confidence < expert.confidence_threshold:
            human_intervention = random.random() > 0.3
        elif acceptance_probability > 0.8:
            human_intervention = random.random() > 0.9
        else:
            human_intervention = random.random() > 0.6

        human_decision = None
        human_confidence = domain_confidence
        if human_intervention:
            human_decision = self._generate_human_decision(
                expert, problem_context, ai_recommendation
            )
            human_confidence = min(1.0, domain_confidence * 1.1)

        self._update_trust(
            expert, ai_confidence, human_intervention, explanation_quality
        )

        return {
            "human_decision": human_decision,
            "human_confidence": human_confidence,
            "human_intervention": human_intervention,
            "explanation_quality": explanation_quality,
            "acceptance_probability": acceptance_probability,
        }

    def _evaluate_explanation(self, explanation: str, expert: HumanExpert) -> float:
        """Return a transparent, rule-based explanation-quality score."""
        quality_factors = [0.8, 0.85, 0.75, 0.7]
        expertise_bonus = {
            ExpertiseLevel.NOVICE: 0.0,
            ExpertiseLevel.INTERMEDIATE: 0.05,
            ExpertiseLevel.EXPERT: 0.1,
            ExpertiseLevel.MASTER: 0.15,
        }
        base_quality = float(np.mean(quality_factors))
        return min(1.0, base_quality + expertise_bonus[expert.expertise_level])

    def _generate_human_decision(
        self,
        expert: HumanExpert,
        problem_context: Dict[str, Any],
        ai_recommendation: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Copy the recommendation before applying any simulated expert adjustment."""
        decision = dict(ai_recommendation)

        if "route" in decision:
            decision["route"] = list(decision["route"])
            adjustment = random.uniform(-0.1, 0.1)
            decision["route"] = self._adjust_route(decision["route"], adjustment)
            customers = problem_context.get("customers", [])
            by_id = {customer["id"]: customer for customer in customers}
            depot = (0, 0)
            locations = [depot] + [by_id[item]["location"] for item in decision["route"]] + [depot]
            decision["total_distance"] = sum(
                self._distance(locations[index], locations[index + 1])
                for index in range(len(locations) - 1)
            )

        if "vehicle_assignment" in decision:
            assignment = {
                vehicle_id: list(customer_ids)
                for vehicle_id, customer_ids in decision["vehicle_assignment"].items()
            }
            if expert.expertise_level == ExpertiseLevel.MASTER:
                assignment = self._optimize_vehicle_assignment(
                    assignment, problem_context
                )
            decision["vehicle_assignment"] = assignment
            customers = problem_context.get("customers", [])
            vehicles = problem_context.get("vehicles", [])
            decision["total_distance"] = self._assignment_distance(
                assignment, customers, vehicles
            )
            decision["vehicles_used"] = sum(bool(ids) for ids in assignment.values())

        return decision

    @staticmethod
    def _distance(first: Tuple[float, float], second: Tuple[float, float]) -> float:
        return float(np.hypot(first[0] - second[0], first[1] - second[1]))

    def _adjust_route(self, route: List[int], adjustment: float) -> List[int]:
        """Apply the model's small random route-order adjustment."""
        if adjustment > 0.05 and len(route) > 2 and random.random() > 0.5:
            first, second = random.sample(range(len(route)), 2)
            route[first], route[second] = route[second], route[first]
        return route

    def _optimize_vehicle_assignment(
        self,
        assignment: Dict[int, List[int]],
        problem_context: Dict[str, Any],
    ) -> Dict[int, List[int]]:
        """Attempt a small workload rebalance without exceeding vehicle capacity."""
        new_assignment = {
            vehicle_id: list(customer_ids)
            for vehicle_id, customer_ids in assignment.items()
        }
        customers_by_id = {
            customer["id"]: customer for customer in problem_context.get("customers", [])
        }
        vehicles_by_id = {
            vehicle["id"]: vehicle for vehicle in problem_context.get("vehicles", [])
        }
        if len(new_assignment) < 2:
            return new_assignment

        workloads = {vehicle_id: len(ids) for vehicle_id, ids in new_assignment.items()}
        overloaded = max(workloads, key=workloads.get)
        underloaded = min(workloads, key=workloads.get)
        if workloads[overloaded] - workloads[underloaded] <= 2:
            return new_assignment
        if not new_assignment[overloaded]:
            return new_assignment

        customer_id = new_assignment[overloaded][-1]
        demand = customers_by_id[customer_id]["demand"]
        destination_load = sum(
            customers_by_id[item]["demand"] for item in new_assignment[underloaded]
        )
        destination_capacity = vehicles_by_id[underloaded]["capacity"]
        if destination_load + demand <= destination_capacity:
            new_assignment[overloaded].pop()
            new_assignment[underloaded].append(customer_id)
        return new_assignment

    def _assignment_distance(
        self,
        assignment: Dict[int, List[int]],
        customers: List[Dict[str, Any]],
        vehicles: List[Dict[str, Any]],
    ) -> float:
        """Measure the assignment's open paths from vehicle starts to customers."""
        customers_by_id = {customer["id"]: customer for customer in customers}
        vehicles_by_id = {vehicle["id"]: vehicle for vehicle in vehicles}
        total_distance = 0.0
        for vehicle_id, customer_ids in assignment.items():
            current_location = vehicles_by_id[vehicle_id]["location"]
            for customer_id in customer_ids:
                next_location = customers_by_id[customer_id]["location"]
                total_distance += self._distance(current_location, next_location)
                current_location = next_location
        return total_distance

    def _update_trust(
        self,
        expert: HumanExpert,
        ai_confidence: float,
        human_intervention: bool,
        explanation_quality: float,
    ) -> None:
        """Update trust using the simulation's stated acceptance and confidence rules."""
        if not human_intervention:
            expert.trust_score = min(
                1.0, expert.trust_score + 0.05 * explanation_quality
            )
        elif ai_confidence > 0.8:
            expert.trust_score = max(0.1, expert.trust_score - 0.03)

        expert.trust_score += random.uniform(-0.01, 0.02)
        expert.trust_score = max(0.1, min(1.0, expert.trust_score))


print("Human expert system defined.")

In [ ]:
class ExplainableAIOptimizer:
    """Produces heuristic recommendations and rule-based confidence scores."""

    def __init__(self, optimization_method: str = "nearest_neighbor_and_greedy_assignment"):
        self.ai = AIOptimizer(
            ai_id="xai_cvrp_001",
            optimization_method=optimization_method,
            confidence_calibration=0.75,
            explanation_capability=0.8,
            learning_capability=0.7,
        )
        self.learning_rate = 0.1
        self.explanation_templates = self._initialize_explanations()

    def _initialize_explanations(self) -> Dict[str, List[str]]:
        """Provide text that describes the implemented heuristic, not an optimality claim."""
        return {
            "route_planning": [
                "The route repeatedly selects the nearest unvisited customer that fits the remaining capacity.",
                "The route follows a nearest-feasible-customer rule and reports any customers it cannot serve.",
            ],
            "vehicle_assignment": [
                "Customers are processed in input order and assigned to the nearest vehicle with sufficient capacity.",
                "The assignment uses vehicle starting locations and checks remaining capacity before assignment.",
            ],
            "priority_adjustment": [
                "This decision type uses the route-planning heuristic; priority labels are not included in its selection rule.",
            ],
            "constraint_handling": [
                "The route-planning heuristic checks vehicle capacity and reports customers left unserved.",
            ],
        }

    def generate_recommendation(self, problem_context: Dict[str, Any]) -> Dict[str, Any]:
        """Generate a recommendation, an illustrative confidence score, and an explanation."""
        decision_type = problem_context.get(
            "decision_type", DecisionType.ROUTE_PLANNING
        )
        if decision_type == DecisionType.ROUTE_PLANNING:
            recommendation = self._solve_route_planning(problem_context)
        elif decision_type == DecisionType.VEHICLE_ASSIGNMENT:
            recommendation = self._solve_vehicle_assignment(problem_context)
        else:
            recommendation = self._solve_general_cvrp(problem_context)

        confidence = self._calculate_confidence(recommendation, problem_context)
        explanation = self._generate_explanation(
            decision_type, recommendation, problem_context
        )
        return {
            "recommendation": recommendation,
            "confidence": confidence,
            "explanation": explanation,
            "reasoning_steps": self._get_reasoning_steps(decision_type),
            "alternatives_considered": [],
        }

    @staticmethod
    def _distance(first: Tuple[float, float], second: Tuple[float, float]) -> float:
        return float(np.hypot(first[0] - second[0], first[1] - second[1]))

    def _solve_route_planning(self, problem_context: Dict[str, Any]) -> Dict[str, Any]:
        """Build one capacity-feasible route by repeatedly choosing the nearest eligible customer."""
        customers = problem_context.get("customers", [])
        capacity = problem_context.get("vehicle_capacity", 15)
        if capacity <= 0:
            raise ValueError("Vehicle capacity must be positive.")
        if any(customer["demand"] < 0 for customer in customers):
            raise ValueError("Customer demands must be nonnegative.")

        depot = (0, 0)
        route: List[int] = []
        current_location = depot
        remaining_capacity = capacity
        unvisited = list(customers)

        while unvisited:
            eligible = [
                customer for customer in unvisited
                if customer["demand"] <= remaining_capacity
            ]
            if not eligible:
                break
            nearest_customer = min(
                eligible,
                key=lambda customer: self._distance(
                    current_location, customer["location"]
                ),
            )
            route.append(nearest_customer["id"])
            current_location = nearest_customer["location"]
            remaining_capacity -= nearest_customer["demand"]
            unvisited.remove(nearest_customer)

        customers_by_id = {customer["id"]: customer for customer in customers}
        locations = [depot] + [customers_by_id[item]["location"] for item in route] + [depot]
        total_distance = sum(
            self._distance(locations[index], locations[index + 1])
            for index in range(len(locations) - 1)
        )
        total_demand = sum(customers_by_id[item]["demand"] for item in route)
        unserved_ids = [customer["id"] for customer in unvisited]
        return {
            "route": route,
            "total_distance": total_distance,
            "total_demand": total_demand,
            "vehicle_utilization": total_demand / capacity,
            "unserved_customer_ids": unserved_ids,
        }

    def _solve_vehicle_assignment(
        self, problem_context: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Greedily assign each customer to the nearest vehicle with remaining capacity."""
        customers = problem_context.get("customers", [])
        vehicles = problem_context.get("vehicles", [])
        if any(vehicle["capacity"] < 0 for vehicle in vehicles):
            raise ValueError("Vehicle capacities must be nonnegative.")

        assignment = {vehicle["id"]: [] for vehicle in vehicles}
        loads = {vehicle["id"]: 0 for vehicle in vehicles}
        vehicles_by_id = {vehicle["id"]: vehicle for vehicle in vehicles}
        unassigned = []

        for customer in customers:
            eligible = [
                vehicle for vehicle in vehicles
                if loads[vehicle["id"]] + customer["demand"] <= vehicle["capacity"]
            ]
            if not eligible:
                unassigned.append(customer["id"])
                continue
            nearest_vehicle = min(
                eligible,
                key=lambda vehicle: self._distance(
                    vehicle["location"], customer["location"]
                ),
            )
            vehicle_id = nearest_vehicle["id"]
            assignment[vehicle_id].append(customer["id"])
            loads[vehicle_id] += customer["demand"]

        total_distance = self._calculate_assignment_distance(
            assignment, customers, vehicles
        )
        return {
            "vehicle_assignment": assignment,
            "vehicle_loads": loads,
            "vehicles_used": sum(bool(customer_ids) for customer_ids in assignment.values()),
            "total_distance": total_distance,
            "unserved_customer_ids": unassigned,
        }

    def _solve_general_cvrp(self, problem_context: Dict[str, Any]) -> Dict[str, Any]:
        """Return separate route and assignment views for the general decision types."""
        route_solution = self._solve_route_planning(problem_context)
        assignment_solution = self._solve_vehicle_assignment(problem_context)
        has_vehicles = bool(problem_context.get("vehicles", []))
        unserved = (
            assignment_solution["unserved_customer_ids"]
            if has_vehicles
            else route_solution["unserved_customer_ids"]
        )
        return {
            "routes": route_solution,
            "assignments": assignment_solution,
            "unserved_customer_ids": unserved,
            "combined_solution": {
                "total_vehicles": (
                    assignment_solution["vehicles_used"]
                    if has_vehicles
                    else int(bool(route_solution["route"]))
                ),
                "total_distance": (
                    assignment_solution["total_distance"]
                    if has_vehicles
                    else route_solution["total_distance"]
                ),
                "service_fraction": (
                    1.0 - len(unserved) / len(problem_context["customers"])
                    if problem_context.get("customers")
                    else 1.0
                ),
            },
        }

    def _calculate_assignment_distance(
        self,
        assignment: Dict[int, List[int]],
        customers: List[Dict[str, Any]],
        vehicles: List[Dict[str, Any]],
    ) -> float:
        """Sum open-path distances from each vehicle start through its assigned customers."""
        customers_by_id = {customer["id"]: customer for customer in customers}
        vehicles_by_id = {vehicle["id"]: vehicle for vehicle in vehicles}
        total_distance = 0.0
        for vehicle_id, customer_ids in assignment.items():
            current_location = vehicles_by_id[vehicle_id]["location"]
            for customer_id in customer_ids:
                next_location = customers_by_id[customer_id]["location"]
                total_distance += self._distance(current_location, next_location)
                current_location = next_location
        return total_distance

    def _calculate_confidence(
        self,
        recommendation: Dict[str, Any],
        problem_context: Dict[str, Any],
    ) -> float:
        """Compute a heuristic score; it is not a statistically calibrated probability."""
        base_confidence = float(np.mean(list(self.ai.performance_metrics.values())))
        customer_count = len(problem_context.get("customers", []))
        complexity_factor = max(0.5, 1.0 - max(0, customer_count - 5) * 0.05)
        solution = recommendation
        if "combined_solution" in recommendation:
            distance = recommendation["combined_solution"]["total_distance"]
            unserved = recommendation["unserved_customer_ids"]
        else:
            distance = recommendation.get("total_distance", 0.0)
            unserved = recommendation.get("unserved_customer_ids", [])

        distance_factor = max(0.7, 1.0 - distance / 100.0)
        coverage_factor = (
            1.0 - len(unserved) / customer_count if customer_count else 1.0
        )
        confidence = (
            base_confidence * complexity_factor * distance_factor * coverage_factor
        )
        confidence += random.uniform(-0.05, 0.05)
        return max(0.3, min(0.95, confidence))

    def _generate_explanation(
        self,
        decision_type: DecisionType,
        recommendation: Dict[str, Any],
        problem_context: Dict[str, Any],
    ) -> str:
        """Select a matching explanation and include available summary values."""
        templates = self.explanation_templates.get(decision_type.value, [])
        explanation = random.choice(templates) if templates else "A heuristic recommendation was generated."
        if "combined_solution" in recommendation:
            summary = recommendation["combined_solution"]
            explanation += f" Reported distance: {summary['total_distance']:.1f} coordinate units."
            explanation += f" Served-customer fraction: {summary['service_fraction']:.1%}."
        elif "total_distance" in recommendation:
            explanation += f" Reported distance: {recommendation['total_distance']:.1f} coordinate units."
        if recommendation.get("unserved_customer_ids"):
            explanation += (
                " Customers left unserved: "
                + ", ".join(map(str, recommendation["unserved_customer_ids"]))
                + "."
            )
        return explanation

    def _get_reasoning_steps(self, decision_type: DecisionType) -> List[str]:
        """Expose the steps used by the selected heuristic."""
        if decision_type == DecisionType.ROUTE_PLANNING:
            return [
                "Read customer locations and demands.",
                "Select the nearest unvisited customer that fits remaining capacity.",
                "Stop when no unvisited customer fits and report unserved customers.",
                "Compute the depot-to-depot route distance and vehicle utilization.",
            ]
        if decision_type == DecisionType.VEHICLE_ASSIGNMENT:
            return [
                "Read vehicle locations and capacities.",
                "Process customers in input order.",
                "Assign each customer to the nearest vehicle with enough remaining capacity.",
                "Report unassigned customers and open-path distance.",
            ]
        return [
            "Generate the route-planning and assignment subresults.",
            "Use the applicable subresult for the summary and service fraction.",
            "Keep the two distance measures separate because they may cover the same customers.",
        ]


print("Explainable heuristic optimizer defined.")

In [ ]:
class HumanAIPartnership:
    """Coordinates recommendations, expert decisions, feedback, and summaries."""

    def __init__(self, num_experts: int = 3):
        self.human_system = HumanExpertSystem(num_experts)
        self.ai_optimizer = ExplainableAIOptimizer()
        self.decision_history: List[CollaborativeDecision] = []
        self.partnership_metrics: Optional[PartnershipMetrics] = None

    def make_collaborative_decision(
        self, problem_context: Dict[str, Any]
    ) -> CollaborativeDecision:
        """Obtain an AI recommendation, simulate expert review, and record the outcome."""
        if not self.human_system.experts:
            raise ValueError("At least one human expert is required.")

        ai_result = self.ai_optimizer.generate_recommendation(problem_context)
        ai_recommendation = ai_result["recommendation"]
        expert_index = len(self.decision_history) % len(self.human_system.experts)
        expert = self.human_system.experts[expert_index]
        human_result = self.human_system.evaluate_ai_recommendation(
            expert,
            ai_recommendation,
            ai_result["confidence"],
            ai_result["explanation"],
            problem_context,
        )

        intervened = human_result["human_intervention"]
        final_decision = (
            human_result["human_decision"] if intervened else ai_recommendation
        )
        decision = CollaborativeDecision(
            decision_id=f"decision_{len(self.decision_history) + 1}",
            timestamp=time.time(),
            problem_context=problem_context,
            ai_recommendation=ai_recommendation,
            ai_confidence=ai_result["confidence"],
            ai_explanation=ai_result["explanation"],
            human_input=human_result["human_decision"],
            human_confidence=human_result["human_confidence"],
            final_decision=final_decision,
            agreement_level=0.0 if intervened else 1.0,
            decision_type=problem_context.get(
                "decision_type", DecisionType.ROUTE_PLANNING
            ),
            explanation_quality=human_result["explanation_quality"],
        )
        self.decision_history.append(decision)
        expert.decision_history.append(decision)
        expert.ai_interactions.append(human_result)
        self._ai_learning_update(human_result, ai_result)
        return decision

    def _ai_learning_update(
        self,
        human_result: Dict[str, Any],
        ai_result: Dict[str, Any],
    ) -> None:
        """Update bounded feedback proxies; acceptance is not treated as verified accuracy."""
        metrics = self.ai_optimizer.ai.performance_metrics
        if not human_result["human_intervention"]:
            metrics["accuracy"] += 0.01
            metrics["explainability"] += 0.005
        else:
            metrics["accuracy"] -= 0.005
            metrics["adaptability"] += 0.01
        for name, value in metrics.items():
            metrics[name] = max(0.3, min(1.0, value))

    def evaluate_partnership(self) -> PartnershipMetrics:
        """Summarize the recorded simulation without inventing benchmark results."""
        if not self.decision_history:
            return PartnershipMetrics(
                total_decisions=0,
                agreement_rate=0.0,
                trust_evolution=[expert.trust_score for expert in self.human_system.experts],
                performance_improvement=None,
                learning_effectiveness=float(
                    np.mean(list(self.ai_optimizer.ai.performance_metrics.values()))
                ),
                explanation_quality=0.0,
                decision_time=None,
                satisfaction_score=0.0,
            )

        decisions = self.decision_history
        agreement_rate = float(np.mean([item.agreement_level for item in decisions]))
        trust_scores = [expert.trust_score for expert in self.human_system.experts]
        explanation_quality = float(
            np.mean([item.explanation_quality for item in decisions])
        )
        learning_effectiveness = float(
            np.mean(list(self.ai_optimizer.ai.performance_metrics.values()))
        )
        # This is a composite simulation index, not a survey-based satisfaction measure.
        satisfaction_score = (agreement_rate + float(np.mean(trust_scores))) / 2.0

        self.partnership_metrics = PartnershipMetrics(
            total_decisions=len(decisions),
            agreement_rate=agreement_rate,
            trust_evolution=trust_scores,
            performance_improvement=None,
            learning_effectiveness=learning_effectiveness,
            explanation_quality=explanation_quality,
            decision_time=None,
            satisfaction_score=satisfaction_score,
        )
        return self.partnership_metrics


print("Human–AI partnership system defined.")

In [ ]:
partnership = HumanAIPartnership(num_experts=3)


def create_test_problems() -> List[Dict[str, Any]]:
    """Create the synthetic routing and assignment instances used in the simulation."""
    customers1 = [
        {"id": 1, "location": (2, 3), "demand": 4},
        {"id": 2, "location": (5, 1), "demand": 3},
        {"id": 3, "location": (8, 6), "demand": 5},
        {"id": 4, "location": (3, 7), "demand": 2},
        {"id": 5, "location": (6, 4), "demand": 3},
    ]
    vehicles = [
        {"id": 1, "location": (0, 0), "capacity": 15},
        {"id": 2, "location": (10, 0), "capacity": 12},
        {"id": 3, "location": (5, 10), "capacity": 18},
    ]
    customers3 = [
        {"id": 1, "location": (1, 2), "demand": 2, "priority": "high"},
        {"id": 2, "location": (4, 3), "demand": 4, "priority": "medium"},
        {"id": 3, "location": (7, 1), "demand": 3, "priority": "low"},
        {"id": 4, "location": (3, 5), "demand": 5, "priority": "high"},
        {"id": 5, "location": (6, 8), "demand": 2, "priority": "medium"},
        {"id": 6, "location": (2, 7), "demand": 4, "priority": "low"},
    ]
    customers4 = [
        {"id": 1, "location": (3, 2), "demand": 8},
        {"id": 2, "location": (6, 5), "demand": 6},
        {"id": 3, "location": (9, 3), "demand": 7},
        {"id": 4, "location": (4, 8), "demand": 5},
    ]

    problems = [
        {
            "decision_type": DecisionType.ROUTE_PLANNING,
            "customers": customers1,
            "vehicle_capacity": 15,
            "description": "Route planning for 5 customers",
        },
        {
            "decision_type": DecisionType.VEHICLE_ASSIGNMENT,
            "customers": customers1,
            "vehicles": vehicles,
            "description": "Vehicle assignment for 5 customers and 3 vehicles",
        },
        {
            "decision_type": DecisionType.PRIORITY_ADJUSTMENT,
            "customers": customers3,
            "vehicle_capacity": 12,
            "description": "Priority-labeled routing instance",
        },
        {
            "decision_type": DecisionType.CONSTRAINT_HANDLING,
            "customers": customers4,
            "vehicle_capacity": 10,
            "description": "Capacity-constrained routing instance",
        },
    ]

    for index in range(6):
        base_problem = random.choice(problems)
        variation = base_problem.copy()
        modified_customers = []
        for customer in variation["customers"]:
            new_customer = customer.copy()
            new_customer["demand"] = max(
                1, customer["demand"] + random.randint(-1, 1)
            )
            modified_customers.append(new_customer)
        variation["customers"] = modified_customers
        variation["description"] = (
            f"Variation {index + 1} of {base_problem['description']}"
        )
        problems.append(variation)

    return problems


test_problems = create_test_problems()
print(f"Created {len(test_problems)} synthetic test problems.")
print(f"Human experts: {[expert.name for expert in partnership.human_system.experts]}")
print(f"AI system identifier: {partnership.ai_optimizer.ai.ai_id}")

## Running the collaboration simulation

The simulation cycles through the supplied problem instances and rotates through the experts. For each instance, the AI first constructs its heuristic recommendation. The expert then accepts it or supplies a simulated adjustment. An intervention counts as disagreement for the agreement indicator, even though the notebook does not have an external answer key with which to judge either decision's quality.

The recommendation logic is intentionally visible in separate methods: `_solve_route_planning`, `_solve_vehicle_assignment`, and `_solve_general_cvrp`. This makes it possible to trace how the problem type determines the calculation and how unserved customers are represented.

In [ ]:
def run_partnership_simulation(
    partnership: HumanAIPartnership,
    problems: List[Dict[str, Any]],
    num_decisions: int = 20,
) -> List[CollaborativeDecision]:
    """Run repeated decisions while preserving the configured expert rotation."""
    if not problems:
        raise ValueError("At least one problem instance is required.")
    if num_decisions < 0:
        raise ValueError("The number of decisions must be nonnegative.")

    decisions = []
    print("Starting the human–AI partnership simulation.")
    print("=" * 60)
    for index in range(num_decisions):
        problem = problems[index % len(problems)]
        decision = partnership.make_collaborative_decision(problem)
        decisions.append(decision)
        expert = partnership.human_system.experts[
            (len(partnership.decision_history) - 1) % len(partnership.human_system.experts)
        ]

        print(f"\nDecision {index + 1}: {problem['description']}")
        print(f"Decision type: {decision.decision_type.value}")
        print(f"Expert: {expert.name} ({expert.expertise_level.value})")
        print(f"AI confidence score: {decision.ai_confidence:.3f}")
        print(f"Human intervention: {'Yes' if decision.human_input is not None else 'No'}")
        print(f"Agreement indicator: {decision.agreement_level:.0f}")
        print(f"Expert trust score: {expert.trust_score:.3f}")
        print(f"AI explanation: {decision.ai_explanation}")
        if decision.final_decision.get("unserved_customer_ids"):
            print(
                "Unserved customer IDs: "
                + ", ".join(
                    map(str, decision.final_decision["unserved_customer_ids"])
                )
            )
        print("-" * 40)

    print(f"\nSimulation recorded {len(decisions)} decisions.")
    return decisions


collaborative_decisions = run_partnership_simulation(
    partnership, test_problems, num_decisions=20
)

## Interpreting the interaction metrics

The agreement rate is the mean of the binary acceptance indicators. Trust scores are the experts' current simulated trust values. Explanation quality is the mean of the rule-based scores used by the expert model. Learning effectiveness is the mean of three bounded feedback proxies stored by the AI object; those values are updated from acceptance or intervention, not from verified routing outcomes.

Performance improvement and decision time are reported as unavailable because this notebook does not measure a benchmark system or elapsed decision duration. The satisfaction score is a simple composite of agreement and final trust; it is not a survey result.

In [ ]:
def analyze_partnership_performance(
    partnership: HumanAIPartnership,
) -> PartnershipMetrics:
    """Display measured simulation summaries and identify unmeasured quantities."""
    metrics = partnership.evaluate_partnership()
    print("\n" + "=" * 60)
    print("HUMAN–AI PARTNERSHIP SIMULATION SUMMARY")
    print("=" * 60)
    print(f"Total decisions: {metrics.total_decisions}")
    print(f"Agreement rate: {metrics.agreement_rate:.1%}")
    print(f"Performance improvement over a benchmark: Not measured")
    print(f"Mean feedback-proxy score: {metrics.learning_effectiveness:.1%}")
    print(f"Mean rule-based explanation-quality score: {metrics.explanation_quality:.1%}")
    print("Decision time: Not measured")
    print(f"Composite satisfaction index (not a survey): {metrics.satisfaction_score:.1%}")

    print("\nFinal simulated trust by expert:")
    for expert in partnership.human_system.experts:
        print(
            f"  {expert.name}: {expert.trust_score:.3f} "
            f"({expert.expertise_level.value})"
        )

    print("\nAI feedback proxies:")
    for metric, value in partnership.ai_optimizer.ai.performance_metrics.items():
        print(f"  {metric.title()}: {value:.1%}")
    return metrics


partnership_metrics = analyze_partnership_performance(partnership)

## Visualizing the recorded decisions

The figures describe the simulation records, rather than proving that the recommendations are optimal or that collaboration improves real-world routing performance. The performance-comparison panel is deliberately marked as unavailable because no independent human-only or AI-only benchmark is provided.

In [ ]:
def create_partnership_visualizations(
    partnership: HumanAIPartnership,
    metrics: PartnershipMetrics,
) -> None:
    """Plot interaction summaries while leaving unmeasured comparisons explicit."""
    decisions = partnership.decision_history
    agreement = [decision.agreement_level for decision in decisions]
    figure, axes = plt.subplots(2, 2, figsize=(14, 10))
    figure.suptitle("Human–AI Partnership Simulation", fontsize=15, fontweight="bold")

    axis = axes[0, 0]
    axis.plot(range(1, len(agreement) + 1), agreement, "o-", color="#2E86AB")
    axis.axhline(
        metrics.agreement_rate,
        color="#A23B72",
        linestyle="--",
        label=f"Mean: {metrics.agreement_rate:.1%}",
    )
    axis.set(title="Recommendation Acceptance", xlabel="Decision number", ylabel="Agreement indicator")
    axis.set_ylim(-0.05, 1.05)
    axis.grid(True, alpha=0.3)
    axis.legend()

    axis = axes[0, 1]
    experts = partnership.human_system.experts
    names = [expert.name.split()[0] for expert in experts]
    trust = [expert.trust_score for expert in experts]
    bars = axis.bar(names, trust, color=["#F18F01", "#C73E1D", "#6A994E"][:len(experts)])
    axis.set(title="Final Simulated Trust by Expert", ylabel="Trust score", ylim=(0, 1))
    for bar, expert in zip(bars, experts):
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            expert.expertise_level.value.title(),
            ha="center",
            va="bottom",
            fontsize=8,
        )

    axis = axes[1, 0]
    ai_metrics = partnership.ai_optimizer.ai.performance_metrics
    bars = axis.bar(list(ai_metrics), list(ai_metrics.values()), color="#3A86FF", alpha=0.75)
    axis.set(title="AI Feedback Proxies", ylabel="Proxy score", ylim=(0, 1))
    axis.tick_params(axis="x", rotation=25)
    for bar, value in zip(bars, ai_metrics.values()):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.1%}", ha="center")

    axis = axes[1, 1]
    types = [decision.decision_type.value for decision in decisions]
    counts = pd.Series(types).value_counts() if types else pd.Series(dtype=int)
    if not counts.empty:
        axis.pie(
            counts.values,
            labels=counts.index,
            autopct="%1.1f%%",
            startangle=90,
            colors=["#FF006E", "#FB5607", "#FFBE0B", "#8338EC"][:len(counts)],
        )
    else:
        axis.text(0.5, 0.5, "No decisions recorded", ha="center", va="center")
    axis.set_title("Decision-Type Distribution")
    plt.tight_layout()
    plt.show()

    figure, axes = plt.subplots(2, 2, figsize=(14, 10))
    figure.suptitle("Decision and Feedback Details", fontsize=15, fontweight="bold")

    interventions = [decision.human_input is not None for decision in decisions]
    axis = axes[0, 0]
    if interventions:
        axis.step(
            range(1, len(interventions) + 1),
            [int(value) for value in interventions],
            where="mid",
            color="#D1495B",
        )
    else:
        axis.text(0.5, 0.5, "No decisions recorded", ha="center", va="center")
    axis.set(title="Human Intervention by Decision", xlabel="Decision number", ylabel="Intervention (0/1)", ylim=(-0.1, 1.1))
    axis.grid(True, alpha=0.3)

    axis = axes[0, 1]
    if decisions:
        axis.scatter(
            [decision.ai_confidence for decision in decisions],
            [decision.human_confidence for decision in decisions],
            alpha=0.7,
            color="#2E86AB",
        )
        axis.plot([0, 1], [0, 1], "r--", alpha=0.6, label="Equal scores")
        axis.legend()
    else:
        axis.text(0.5, 0.5, "No decisions recorded", ha="center", va="center")
    axis.set(title="AI and Human Confidence Scores", xlabel="AI heuristic score", ylabel="Human domain score", xlim=(0, 1), ylim=(0, 1))
    axis.grid(True, alpha=0.3)

    axis = axes[1, 0]
    axis.axis("off")
    axis.set_title("Benchmark Comparison")
    axis.text(
        0.5,
        0.5,
        "Not available\nNo independent benchmark data were supplied.",
        ha="center",
        va="center",
        fontsize=11,
        transform=axis.transAxes,
    )

    axis = axes[1, 1]
    if agreement:
        window_size = min(5, len(agreement))
        moving_average = [
            float(np.mean(agreement[max(0, index - window_size + 1):index + 1]))
            for index in range(len(agreement))
        ]
        axis.plot(range(1, len(agreement) + 1), agreement, color="#2E86AB", alpha=0.35, label="Individual decisions")
        axis.plot(range(1, len(moving_average) + 1), moving_average, color="#2A9D8F", linewidth=2, label=f"Moving mean (window={window_size})")
        axis.legend()
    else:
        axis.text(0.5, 0.5, "No decisions recorded", ha="center", va="center")
    axis.set(title="Agreement Indicator Over Time", xlabel="Decision number", ylabel="Agreement indicator", ylim=(-0.05, 1.05))
    axis.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


create_partnership_visualizations(partnership, partnership_metrics)

## Summary and interpretation

The summary below reports interaction quantities that the simulation actually records. Its agreement rate describes whether the simulated expert accepted the recommendation. Trust and feedback values follow the notebook's update rules. None of these quantities establishes route optimality, operational performance, or real-world human–AI synergy without external validation.

In [ ]:
def generate_partnership_insights(
    partnership: HumanAIPartnership,
    metrics: PartnershipMetrics,
) -> str:
    """Create a concise interpretation without treating simulation proxies as evidence."""
    decisions = partnership.decision_history
    high_confidence_count = sum(
        decision.ai_confidence > 0.8 for decision in decisions
    )
    intervention_count = sum(
        decision.human_input is not None for decision in decisions
    )

    lines = [
        "\n" + "=" * 72,
        "HUMAN–AI PARTNERSHIP SIMULATION: SUMMARY",
        "=" * 72,
        f"Recorded decisions: {metrics.total_decisions}",
        f"Recommendation acceptance rate: {metrics.agreement_rate:.1%}",
        f"Mean rule-based explanation-quality score: {metrics.explanation_quality:.1%}",
        f"Mean AI feedback-proxy score: {metrics.learning_effectiveness:.1%}",
        "Benchmark performance improvement: Not measured",
        "Decision time: Not measured",
        "",
        "Decision review:",
    ]

    if decisions:
        lines.append(
            f"High-confidence AI scores: {high_confidence_count}/{len(decisions)} "
            f"({high_confidence_count / len(decisions):.1%})"
        )
        lines.append(
            f"Human interventions: {intervention_count}/{len(decisions)} "
            f"({intervention_count / len(decisions):.1%})"
        )
    else:
        lines.append("No decisions were recorded.")

    lines.extend(["", "Final simulated trust by expert:"])
    for expert in partnership.human_system.experts:
        lines.append(
            f"{expert.name}: {expert.trust_score:.3f} "
            f"({expert.expertise_level.value})"
        )

    lines.extend(
        [
            "",
            "Interpretation:",
            "Acceptance, trust, confidence, and feedback values are generated by the stated simulation rules.",
            "They are not measurements of decision correctness or evidence of performance gains.",
            "The routing heuristics may leave customers unserved; inspect each recommendation's coverage and constraints.",
        ]
    )
    return "\n".join(lines)


insights = generate_partnership_insights(partnership, partnership_metrics)
print(insights)

## Conclusion

This notebook demonstrates a rule-based human–AI interaction framework around CVRP-related decisions. Its visible sequence is: generate a heuristic recommendation, calculate an illustrative confidence score, evaluate the recommendation with a simulated expert, record acceptance or intervention, and update trust and feedback proxies.

The routing components are deliberately limited. The single-vehicle route planner uses nearest-feasible-customer selection, while the vehicle-assignment component uses a greedy nearest-eligible-vehicle rule. Neither is an exact CVRP algorithm, and neither guarantees a globally shortest solution. The code reports unserved customers so that feasibility and coverage remain inspectable.

The interaction metrics describe this simulation only. In particular, acceptance is not correctness, confidence is not calibration, and feedback-proxy changes are not verified learning. Demonstrating real performance improvement would require an independent benchmark, a defined objective and evaluation protocol, and validated outcome data. Real deployments would also need operational constraints such as time windows, service times, road-network travel costs, and changing conditions.